# Narrador IA — ligar o servidor (GPU, via Kaggle)

Alternativa ao `colab_gerar_narracao.ipynb` para quando o Colab estiver com a cota gratuita de GPU esgotada (mensagem "limites de uso gratuito não são suficientes" / sem opção de GPU). O Kaggle tem uma cota **separada** (por volta de 30h de GPU por semana), então funciona como plano B enquanto a cota do Colab não libera.

Assim como no Colab, este notebook só liga o "motor" — depois de rodar, você recebe um link e **toda a interação real acontece na página web** que ele expõe.

**Antes de rodar**, no painel à direita do editor do Kaggle:
1. **Accelerator → GPU T4 x2** (ou P100)
2. **Internet → On** (sem isso, git clone/pip install/ngrok não funcionam)

**Depois:** menu **Run → Run All**.

**Não rode este notebook ao mesmo tempo que o do Colab** com o mesmo `NGROK_TOKEN` — o plano grátis do ngrok só permite 1 túnel por vez; a célula do servidor já tenta lidar com esse conflito, mas evite rodar os dois juntos de propósito.

In [ ]:
import os
import subprocess

sPastaProjeto = "/kaggle/working/narrador"

if os.path.isdir(sPastaProjeto):
    # reset --hard (em vez de pull) porque o histórico remoto pode ter sido
    # reescrito (force-push) desde o último clone nesta sessão — pull
    # falharia tentando reconciliar históricos sem ancestral comum.
    subprocess.run(["git", "-C", sPastaProjeto, "fetch", "origin"], check=True)
    subprocess.run(
        ["git", "-C", sPastaProjeto, "reset", "--hard", "origin/prod"], check=True
    )
    subprocess.run(["git", "-C", sPastaProjeto, "clean", "-fd"], check=True)
else:
    subprocess.run(
        ["git", "clone", "https://github.com/guzsysdev/narrador.git", sPastaProjeto],
        check=True,
    )

os.chdir(sPastaProjeto)

# O Kaggle já vem com PyTorch + CUDA — não reinstalamos torch para não quebrar isso.
subprocess.run(
    ["pip", "install", "-q", "voxcpm", "librosa", "soundfile",
     "fastapi", "uvicorn", "pydantic", "python-multipart", "pyngrok"],
    check=True,
)

import torch
print("GPU disponível:", torch.cuda.is_available())
print("Dispositivo:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA — ative GPU no painel Accelerator, à direita")


### Token do ngrok (só a primeira vez)

Pra expor o servidor numa URL pública, usamos o ngrok (grátis) — o mesmo token que você já usa no Colab funciona aqui também. Pra nunca mais ter que colar o token:

1. Se ainda não tiver, crie uma conta grátis em https://ngrok.com e pegue seu authtoken em https://dashboard.ngrok.com/get-started/your-authtoken
2. No editor do Kaggle, menu **Add-ons → Secrets**
3. Adicione um novo secret: **Label** `NGROK_TOKEN`, **Value** = seu authtoken
4. Marque a chavinha "Attached" pra esse secret ficar disponível neste notebook

Feito isso uma vez, a célula abaixo encontra o token sozinha em qualquer sessão futura. Se não configurar, ela vai pedir pra colar manualmente (também funciona, só que toda vez).

In [ ]:
sTokenNgrok = None
try:
    from kaggle_secrets import UserSecretsClient
    sTokenNgrok = UserSecretsClient().get_secret("NGROK_TOKEN")
except Exception:
    pass

if not sTokenNgrok:
    from getpass import getpass
    sTokenNgrok = getpass("Secret NGROK_TOKEN não encontrado — cole seu ngrok authtoken: ")


### Token do GitHub (opcional — só se quiser usar "Salvar permanentemente")

Na interface web, ao gerar opções de voz aleatórias, tem um botão "Salvar permanentemente" que publica a voz escolhida direto no catálogo do GitHub. Isso só funciona com um token do GitHub configurado aqui — o mesmo `GITHUB_TOKEN` fine-grained que você já usa no Colab funciona aqui também.

1. Se ainda não tiver, crie um **fine-grained personal access token** em https://github.com/settings/personal-access-tokens/new (repositório só `guzsysdev/narrador`, permissão só **Contents: Read and write**)
2. No editor do Kaggle, **Add-ons → Secrets** → adicione **Label** `GITHUB_TOKEN`, **Value** = o token
3. Marque "Attached"

Se você não quiser usar essa função, pode pular esta célula — o resto do notebook funciona normalmente sem o token.

In [ ]:
sTokenGithub = None
try:
    from kaggle_secrets import UserSecretsClient
    sTokenGithub = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass

if not sTokenGithub:
    print('Secret GITHUB_TOKEN não encontrado — a função "Salvar permanentemente" '
          'não vai funcionar (o resto do notebook funciona normalmente).')


In [ ]:
import time
import json as jsonlib
import urllib.request
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokHTTPError

ngrok.set_auth_token(sTokenNgrok)
ngrok.kill()  # encerra qualquer túnel LOCAL desta sessão (não afeta outra sessão)

# Repassa o GITHUB_TOKEN (se configurado) pro ambiente do servidor, pra que
# o botão "Salvar permanentemente" na interface web consiga commitar/publicar.
oAmbienteServidor = os.environ.copy()
if sTokenGithub:
    oAmbienteServidor["GITHUB_TOKEN"] = sTokenGithub

oProcessoServidor = subprocess.Popen(
    ["uvicorn", "servidorWeb:oApp", "--host", "0.0.0.0", "--port", "8000"],
    env=oAmbienteServidor,
)

print("Aguardando o modelo carregar (alguns minutos na primeira vez)...", end="", flush=True)
iTentativas = 0
iTentativasMaximas = 180  # ~15 min (180 x 5s) — depois disso, algo está errado
while iTentativas < iTentativasMaximas:
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/api/saude", timeout=3) as oResp:
            oStatus = jsonlib.loads(oResp.read())
            if oStatus.get("sErro"):
                print()
                raise RuntimeError(f"O modelo falhou ao carregar: {oStatus['sErro']}")
            if oStatus.get("lPronto"):
                break
    except urllib.error.URLError:
        pass  # servidor ainda subindo, tenta de novo
    print(".", end="", flush=True)
    time.sleep(5)
    iTentativas += 1
else:
    raise TimeoutError(
        "Modelo não ficou pronto em 15 min. Role para cima nesta célula e procure "
        "por um traceback no log do uvicorn (pode ter travado sem erro explícito)."
    )
print()

# ERR_NGROK_334 ("endpoint already online") acontece quando outra sessão (aba
# antiga do Colab ou do Kaggle, ainda aberta ou não totalmente encerrada) está
# com esse mesmo túnel ativo — o plano grátis do ngrok só permite 1 túnel por
# vez. Tentamos de novo algumas vezes; se persistir, é preciso encerrar a
# outra sessão manualmente.
oTunel = None
for iTentativaTunel in range(5):
    try:
        oTunel = ngrok.connect(8000)
        break
    except PyngrokNgrokHTTPError as oErro:
        if "already online" not in str(oErro):
            raise
        print(f"Túnel já em uso por outra sessão (Colab ou Kaggle), tentando de novo em 15s "
              f"({iTentativaTunel + 1}/5)...")
        time.sleep(15)

if oTunel is None:
    raise RuntimeError(
        "Não consegui abrir o túnel: outra sessão (Colab ou Kaggle) ainda está usando o "
        "mesmo link do ngrok. Encerre a outra sessão, ou pare o túnel manualmente em "
        "https://dashboard.ngrok.com/agents — depois rode esta célula de novo."
    )

print("=" * 60)
print("Modelo pronto! Abra esta URL no navegador:", oTunel.public_url)
print("=" * 60)

# Mantém esta célula "ocupada" de propósito: sessões interativas do Kaggle
# também desconectam por inatividade quando nenhuma célula está executando —
# e sem isso, a célula terminaria assim que imprimisse a URL acima, mesmo com
# o servidor (uvicorn) ainda rodando em background. Este loop finge
# atividade pra manter o kernel "ocupado" de verdade.
#
# Isso NÃO evita o limite absoluto de sessão nem a cota semanal de GPU do
# Kaggle — só evita a desconexão por ociosidade.
#
# Pra encerrar: use o painel de sessão do Kaggle (canto superior direito do
# editor, ícone de energia/"Stop Session") — não afeta o servidor até você
# parar a sessão de vez por lá.
print()
print("Servidor ativo — mantendo a sessão viva (pare a sessão pelo painel do Kaggle "
      "quando quiser desligar tudo).")
iMinutosAtivo = 0
while True:
    time.sleep(60)
    iMinutosAtivo += 1
    print(f"[{iMinutosAtivo} min] servidor ativo — {oTunel.public_url}")


## Pronto

Abra o link acima no navegador — é lá que você faz tudo: escolhe a voz, ouve a prévia, cola o roteiro completo, ajusta velocidade/tom, gera e baixa o WAV final.

A URL muda toda vez que a célula acima é executada de novo — não compartilhe, qualquer pessoa com o link consegue usar o servidor enquanto ele estiver no ar.

Quando a cota de GPU do Colab liberar de novo, pode voltar a usar `colab_gerar_narracao.ipynb` normalmente — os dois notebooks usam o mesmo repositório e catálogo de vozes, então nada se perde trocando de um pro outro.

### Pra encerrar o servidor mais tarde

NÃO adicione uma célula de `ngrok.disconnect()`/`terminate()` aqui — se você rodar "Run All" de novo, ela rodaria logo em seguida e derrubaria o servidor que acabou de subir. Pra encerrar, use o painel de sessão do Kaggle (ícone de energia / "Stop Session", no canto superior direito do editor) — isso mata o servidor e o túnel de forma limpa.